# WAF Gen 3 — เทรนและทดลองบน Google Colab

Notebook นี้รัน pipeline Gen 3 จาก branch `trainmodelgen3` แล้วเก็บผลกลับมาให้วิเคราะห์ต่อบนเครื่อง local

| ขั้น | สคริปต์ | เวลาโดยประมาณ (Colab ฟรี, 1 physical core) |
| :--- | :--- | :--- |
| 1. ทดลองฟีเจอร์ระดับค่า (value features) — **งานหลักรอบนี้** | `ml/experiment_value_features.py` | 30–60 นาที (+ LOSO อีก ~1 ชม.) |
| 2. เทรน hybrid candidate ใหม่ | `ml/train_gen3_full_real_benchmark.py` | 1–2 ชม. |
| 3. โมเดลราย unit (MIL) + stress test | `ml/experiment_unit_model.py` | 1–2 ชม. (ต้องรันขั้น 2 ก่อน) |

ครั้งแรกจะใช้เวลาเพิ่มอีก ~10 นาทีสำหรับดาวน์โหลด public dataset (~1.7 GB) และสร้าง dataset — ถ้าเปิด `USE_DRIVE` จะเก็บ cache ไว้ใน Google Drive ให้รอบต่อไปเร็วขึ้น

**ข้อควรระวัง**
- ใช้ Colab ฟรีได้ทั้งหมด ไม่ต้องใช้ GPU (LightGBM/sklearn ใช้ CPU) — ถ้า runtime หลุดระหว่างรัน ให้รัน cell เดิมซ้ำ (cache ใน Drive ทำให้ไม่ต้องเริ่มใหม่ทั้งหมด)
- พื้นที่ Google Drive ที่ใช้ ~3–4 GB (ฟรี 15 GB)
- ข้อมูลจาก VPS (`ml/dataset/*.jsonl`, `ml/telemetry/`) เป็น traffic จริงจาก production — **ใส่ก็ต่อเมื่อได้รับอนุญาตให้เก็บบน Google Drive แล้วเท่านั้น** ถ้าไม่ใส่ notebook จะรันด้วย public data (CSIC + open-appsec + SR-BH) และ gate CORE จะเหลือแค่ CSIC ซึ่งเทียบตรงๆ กับ candidate เดิมไม่ได้
- ห้ามแตะ VPS production จาก notebook นี้

In [ ]:
# ===== ตั้งค่า =====
REPO = "jakkaret/waf_project"          # GitHub owner/repo
BRANCH = "trainmodelgen3"
USE_DRIVE = True                        # เก็บ dataset/cache/ผลลัพธ์ไว้ใน Google Drive
DRIVE_DIR = "/content/drive/MyDrive/waf_ml"
# ไฟล์ข้อมูล VPS แบบ private (สร้างตาม ml/TRAINING_SETUP.md) — วางไว้ที่ DRIVE_DIR/private/ หรือปล่อยว่างเพื่อใช้ public data อย่างเดียว
PRIVATE_TARBALL_GLOB = DRIVE_DIR + "/private/waf_ml_private_data_*.tar.gz"

RUN_VALUE_FEATURE_EXPERIMENT = True
VALUE_EXPERIMENT_LOSO = True            # leave-one-dataset-out (ช้าที่สุด แต่เป็นตัวชี้วัดความทนทานที่สำคัญ)
VALUE_EXPERIMENT_ONLY_LOFO = True       # รันเฉพาะ leave-one-dataset-out (~15 นาที) — ตั้ง False เพื่อรันครบทุก fold
RUN_TRAINER = False                     # เทรน hybrid candidate ใหม่ (ต้องใช้ก่อน RUN_UNIT_EXPERIMENT)
RUN_UNIT_EXPERIMENT = False

import os, time, glob
REPO_DIR = "/content/waf_project"
VENV_PY = "/content/venv/bin/python"
SESSION_START = time.time()
os.environ["PYTHONUNBUFFERED"] = "1"   # log ไหลออกทันทีแม้ผ่าน tee
print("CPU:", os.cpu_count(), "| RAM GB:", round(os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1e9, 1))

In [ ]:
# ===== Google Drive + clone repo =====
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_DIR, exist_ok=True)

# repo private: เพิ่ม Colab Secret ชื่อ GITHUB_TOKEN (fine-grained token, สิทธิ์ Contents: read) ที่แถบ 🔑 ด้านซ้าย
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"

if not os.path.isdir(REPO_DIR + "/.git"):
    !git clone -q --branch {BRANCH} {url} {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch -q origin {BRANCH} && git -C {REPO_DIR} checkout -q {BRANCH} && git -C {REPO_DIR} reset -q --hard origin/{BRANCH}
!git -C {REPO_DIR} remote set-url origin https://github.com/{REPO}.git
!git -C {REPO_DIR} log --oneline -3
os.chdir(REPO_DIR)
os.makedirs("colab_logs", exist_ok=True)

# คัดลอก log ขึ้น Drive ทุก 2 นาที (MyDrive/waf_ml/live_logs/) ให้ Claude ตามดูความคืบหน้าผ่าน Google Drive ได้
import threading, shutil
LIVE_LOG_DIR = f"{DRIVE_DIR}/live_logs"
def _sync_logs():
    while True:
        try:
            os.makedirs(LIVE_LOG_DIR, exist_ok=True)
            for f in glob.glob(f"{REPO_DIR}/colab_logs/*.log"):
                shutil.copy2(f, LIVE_LOG_DIR)
            with open(f"{LIVE_LOG_DIR}/heartbeat.txt", "w") as fp:
                fp.write(time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime()) + "\n")
        except Exception:
            pass
        time.sleep(120)
if USE_DRIVE and not globals().get("_log_sync_started"):
    threading.Thread(target=_sync_logs, daemon=True).start()
    _log_sync_started = True

In [ ]:
# ===== Python 3.12 venv แยกจาก Colab (เวอร์ชันตรงกับที่ใช้เทรน candidate เดิม) =====
!pip -q install uv 2>/dev/null; uv --version
if not os.path.exists(VENV_PY):  # รันซ้ำใน runtime เดิม: uv venv จะถาม "replace it? [y/n]" แล้วค้างรอ input
    !uv venv -q --python 3.12 /content/venv
!uv pip install -q --python {VENV_PY} -r {REPO_DIR}/ml/requirements-train.txt
!{VENV_PY} -c "import sys, lightgbm, sklearn, libinjection; print(sys.version.split()[0], lightgbm.__version__, sklearn.__version__)"

In [ ]:
# ===== เก็บ dataset / cache ไว้ใน Drive (symlink) =====
import shutil
def persist(rel):
    local, remote = f"{REPO_DIR}/{rel}", f"{DRIVE_DIR}/{rel.replace('/', '__')}"
    os.makedirs(remote, exist_ok=True)
    if os.path.islink(local):
        return
    if os.path.isdir(local):
        shutil.copytree(local, remote, dirs_exist_ok=True)
        shutil.rmtree(local)
    os.makedirs(os.path.dirname(local), exist_ok=True)
    os.symlink(remote, local)

if USE_DRIVE:
    for rel in ("ml/dataset/external", "ml/dataset/.build_cache"):
        persist(rel)
    csic = f"{DRIVE_DIR}/csic_final.csv"
    if os.path.exists(csic) and not os.path.exists(f"{REPO_DIR}/ml/dataset/csic_final.csv"):
        !cp {csic} {REPO_DIR}/ml/dataset/
!ls -la {REPO_DIR}/ml/dataset

In [ ]:
# ===== ข้อมูล VPS แบบ private (ถ้ามี) =====
tarballs = sorted(glob.glob(PRIVATE_TARBALL_GLOB)) if USE_DRIVE else []
if tarballs:
    print("แตกไฟล์:", tarballs[-1])
    !tar -xzf "{tarballs[-1]}" -C {REPO_DIR}
else:
    print("⚠️ ไม่มีข้อมูล VPS — รันด้วย public data อย่างเดียว (gate CORE = CSIC เท่านั้น)")
for rel in ("ml/dataset/vps_audit_labelled.jsonl", "ml/dataset/nginx_real_benign.jsonl", "ml/telemetry"):
    print("✅" if os.path.exists(f"{REPO_DIR}/{rel}") else "—", rel)

In [ ]:
# ===== Public dataset: open-appsec + SR-BH 2020 (ตรวจ sha256) — ข้ามถ้าเคยเตรียมไว้แล้ว =====
if not os.path.exists("ml/dataset/external/prepare_stats.json"):
    !PYTHONPATH=. {VENV_PY} ml/prepare_external_datasets.py --download 2>&1 | tee colab_logs/prepare_external.log
!ls -la ml/dataset/external | head -20
# หยุดที่นี่ถ้าเตรียมไม่สำเร็จ — ถ้ารันต่อจะได้ผลจาก CSIC อย่างเดียวซึ่งใช้ตัดสินเรื่องฟีเจอร์ไม่ได้
assert os.path.exists("ml/dataset/external/prepare_stats.json"), "เตรียม public dataset ไม่สำเร็จ — ดู colab_logs/prepare_external.log"
if USE_DRIVE and os.path.exists("ml/dataset/csic_final.csv"):
    !cp -n ml/dataset/csic_final.csv {DRIVE_DIR}/

In [ ]:
# ===== Unit tests =====
!PYTHONPATH=. {VENV_PY} -m pytest ml/test_value_features.py ml/test_feature_engineering.py ml/tests -q -p no:warnings 2>&1 | tail -5 | tee colab_logs/unit_tests.log

In [ ]:
# ===== 1. ทดลอง value features: A (36 เดิม) / B (+value) / E (ตัดฟีเจอร์บริบท) / E_plus_* (ablation ทีละฟีเจอร์) =====
if RUN_VALUE_FEATURE_EXPERIMENT:
    loso = "--only-lofo" if VALUE_EXPERIMENT_ONLY_LOFO else ("" if VALUE_EXPERIMENT_LOSO else "--no-loso")
    !PYTHONPATH=. {VENV_PY} -W ignore ml/experiment_value_features.py {loso} 2>&1 | tee colab_logs/value_features.log

In [ ]:
# ===== 2. เทรน hybrid candidate (hyperparameter grid + holdout) =====
if RUN_TRAINER:
    !PYTHONPATH=. {VENV_PY} -W ignore ml/train_gen3_full_real_benchmark.py 2>&1 | tee colab_logs/train_gen3.log
    !PYTHONPATH=. {VENV_PY} -W ignore ml/test_comprehensive.py 2>&1 | tail -15 | tee colab_logs/scenarios.log

In [ ]:
# ===== 3. โมเดลราย unit (MIL) + context-transfer stress test + LOSO =====
if RUN_UNIT_EXPERIMENT:
    !PYTHONPATH=. {VENV_PY} -W ignore ml/experiment_unit_model.py 2>&1 | tee colab_logs/unit_model.log

In [ ]:
# ===== สรุปผลการทดลอง value features =====
import json, pandas as pd
reports = sorted(glob.glob("ml/models/archive/experiment-value-features-*/value_feature_report.json"))
if reports:
    rep = json.load(open(reports[-1]))
    fmt = lambda d: f"{d['mean']*100:.1f} ± {d['std']*100:.1f}" if d else None
    rows, lofo = [], []
    for name, e in rep["configs"].items():
        s = e.get("summary") or {}  # empty on an --only-lofo run
        if s:
            rows.append({"config": name, "features": e["n_features"], "CORE benign": fmt(s["core_benign"]),
                         "CORE attack": fmt(s["core_attack"]), "LFI": fmt(s["lfi"]), "SQLi": fmt(s["sqli"]),
                         "stress detect": fmt(s["stress_detect"]), "stress FP": fmt(s["stress_control_fp"]),
                         "scenario attack": s["scenario_attack_blocked"]["mean"], "scenario normal": s["scenario_normal_allowed"]["mean"]})
        for held, r in e.get("leave_one_dataset_out", {}).items():
            t = r["at_training_threshold"]
            lofo.append({"config": name, "held out": held, "AUC": r.get("auc"),
                         "attack @ benign 98.5%": r.get("attack_recall_at_benign_98_5"),
                         "attack @ benign 99.9%": r.get("attack_recall_at_benign_99_9"),
                         "training-thr benign": t["benign_recall"], "training-thr attack": t["attack_recall"]})
    print(reports[-1], "| sources:", rep["sources"], "| holdout folds:", rep.get("holdout_folds"))
    if rows:
        display(pd.DataFrame(rows).set_index("config"))
    if lofo:
        display(pd.DataFrame(lofo).set_index(["held out", "config"]).sort_index())

In [ ]:
# ===== รวมผลเป็น zip (รายงาน JSON + log; ไม่รวมไฟล์โมเดล .joblib ซึ่งใหญ่) =====
stamp = time.strftime("%Y%m%d-%H%M%S")
new_dirs = [d for d in glob.glob("ml/models/archive/*") if os.path.isdir(d) and os.path.getmtime(d) >= SESSION_START]
print("ผลลัพธ์รอบนี้:", new_dirs)
out = f"/content/waf_colab_results_{stamp}.zip"
targets = " ".join(new_dirs + ["colab_logs"])
!zip -qr {out} {targets} -x "*.joblib"
if USE_DRIVE:
    os.makedirs(f"{DRIVE_DIR}/results", exist_ok=True)
    !cp {out} {DRIVE_DIR}/results/
    # รายงาน JSON + log แบบไม่บีบอัด ให้ Claude อ่านผ่าน Google Drive ได้โดยตรง
    run_dir = f"{DRIVE_DIR}/results/{stamp}"
    for d in new_dirs + ["colab_logs"]:
        shutil.copytree(d, f"{run_dir}/{os.path.basename(d)}", ignore=shutil.ignore_patterns("*.joblib"), dirs_exist_ok=True)
    print("บันทึกผลไว้ที่", run_dir)
    # โมเดลเก็บไว้ใน Drive (ไม่ขึ้น git)
    for d in new_dirs:
        for f in glob.glob(f"{d}/*.joblib"):
            os.makedirs(f"{DRIVE_DIR}/models/{os.path.basename(d)}", exist_ok=True)
            shutil.copy2(f, f"{DRIVE_DIR}/models/{os.path.basename(d)}/")
try:
    from google.colab import files
    files.download(out)
except Exception as exc:  # ผลอยู่ใน Drive แล้ว ไม่ต้องดาวน์โหลดก็ได้
    print("ดาวน์โหลดอัตโนมัติไม่สำเร็จ:", exc)

## เอาผลกลับไปวิเคราะห์

ผลอยู่ใน Google Drive ที่ `MyDrive/waf_ml/results/<วันเวลา>/` แล้ว — ถ้า Claude ใน VS Code เชื่อม Google Drive ไว้ บอกได้เลยว่า *"วิเคราะห์ผล value features ล่าสุด"* ไม่ต้องดาวน์โหลดเอง
(หรือแตก zip ที่โฟลเดอร์ root ของ repo บนเครื่อง local ไฟล์จะลงที่ `ml/models/archive/<experiment>/` และ `colab_logs/`)

เกณฑ์ตัดสินว่า config ไหนดีขึ้นจริง (ไม่ใช่แค่จำ dataset):
1. **CORE gate** — benign ≥ 98.5% และ attack ≥ 85% (holdout, threshold จาก OOF)
2. **stress detect** สูงขึ้นโดย **stress FP** ไม่เพิ่ม — payload เดิมในบริบทใหม่ยังถูกจับได้
3. **scenario attack** ≥ 23/26 โดย normal ยัง 37/37
4. **LOSO attack** สูงขึ้น — ทนต่อ dataset ที่ไม่เคยเห็น

ถ้า `attack structural-only` ต่ำทุก config แสดงว่าเป็น CSIC parameter tampering ที่ไม่มีสัญญาณโจมตีใน request — ฟีเจอร์ช่วยไม่ได้ ต้องตัดสินเรื่องนิยาม gate (ดู `PROGRESS_GEN3_CHECKPOINT.md`)